# 주간 모델 학습 노트북

**매주 1회 돌리는 노트북**입니다. 3개 모델을 모두 재학습합니다.

### 앱별 모델 분리
앱별로 모델이 다릅니다. `APP_ID`를 바꾸면 다른 앱의 모델을 학습할 수 있어요.

### 이 노트북이 하는 일
1. 데이터 로드 (`weekly_data.csv` — 이번 주 신규 유저)
2. Feature 정의 (UA 15개 + In-app 10개 = 25개)
3. Factor Analysis → `fa_params.json` (유저 성향 차원 추출)
4. D3 Purchase 모델 → `d3_purchase_model.pkl` (전체 유저로 학습)
5. D3 Churn 모델 → `d3_churn_model.pkl` (전체 유저로 학습)
6. CATE 모델 → `cate_model.pkl` (**랜덤 배정 유저만** 사용)
7. 최종 확인 & 테스트

### 만들어지는 파일
| 파일 | 설명 | 업데이트 빈도 |
|------|------|-------------|
| `models/{APP_ID}/fa_params.json` | FA 파라미터 (유저 성향 차원) | 거의 안 바뀜 |
| `models/{APP_ID}/d3_purchase_model.pkl` | D3 구매 예측 | 매주 |
| `models/{APP_ID}/d3_churn_model.pkl` | D3 이탈 예측 | 매주 |
| `models/{APP_ID}/cate_model.pkl` | Trigger 반응 예측 (CATE) | 매주 |

### 데이터 구조
- **전체 유저**: UA features + In-app features + d3_purchase + d3_churn
- **80% 모델 추천 유저**: `is_random=0` — 모델이 best trigger를 배정. 학습에 편향 있음.
- **20% 랜덤 유저**: `is_random=1` — 랜덤 배정. **CATE 학습에만 이 유저를 사용.**

In [ ]:
# Supabase에서 prediction 로그 다운로드 (선택 — 실서비스에서 사용)
# from supabase import create_client
# client = create_client(SUPABASE_URL, SUPABASE_KEY)
# logs = client.table("prediction_logs").select("*").eq("app_id", APP_ID).execute()
# df_logs = pd.DataFrame(logs.data)
# 
# CATE 학습용: is_random=True만 필터
# df_cate_logs = df_logs[df_logs['is_random'] == True]

---
## Step 1: 데이터 로드

`weekly_data.csv`는 이번 주 신규 유저 1000명의 데이터입니다.

각 유저가 갖고 있는 정보:
- **UA features (15개)**: 앱 설치 전 광고 여정 데이터 (Airbridge DB)
- **In-app features (10개)**: 앱 오픈 후 첫 5분 행동 데이터 (SDK)
- **assigned_trigger**: 배정된 trigger (price_appeal / social_proof / scarcity / novelty)
- **is_random**: 1이면 랜덤 배정, 0이면 모델 추천
- **modal_clicked**: 모달 클릭 여부
- **first_session_purchase**: 첫 세션 구매 여부
- **d3_purchase**: 3일 내 구매 여부
- **d3_churn**: 3일 내 이탈 여부

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 앱별로 모델이 다릅니다. APP_ID를 바꾸면 다른 앱의 모델을 학습할 수 있어요.
APP_ID = "ablog"

# 앱별 모델 저장 디렉토리 생성
os.makedirs(f'../models/{APP_ID}', exist_ok=True)

# 데이터 로드
df = pd.read_csv('../data/weekly_data.csv')

# app_id 컬럼이 있으면 해당 앱만 필터링
if 'app_id' in df.columns:
    df = df[df['app_id'] == APP_ID].reset_index(drop=True)

print(f'앱: {APP_ID}')
print(f'총 유저 수: {len(df)}')
print(f'랜덤 배정 (is_random=1): {df["is_random"].sum()} ({df["is_random"].mean():.1%})')
print(f'모델 추천 (is_random=0): {(1-df["is_random"]).sum():.0f} ({(1-df["is_random"]).mean():.1%})')
print(f'Organic 유저 (UA=0): {(df["trackinglink_count"] == 0).sum()}명')
print(f'D3 구매율: {df["d3_purchase"].mean():.1%}')
print(f'D3 이탈율: {df["d3_churn"].mean():.1%}')
print()
print('Trigger 배정:')
print(df['assigned_trigger'].value_counts())
print()
print('모달 클릭률:')
print(f'  랜덤 유저: {df[df["is_random"]==1]["modal_clicked"].mean():.1%}')
print(f'  모델 유저: {df[df["is_random"]==0]["modal_clicked"].mean():.1%}')
print(f'  → 모델 유저가 더 높으면 모델이 잘 작동하는 것!')

---
## Step 2: Feature 정의

서비스에서 사용하는 feature는 총 25개입니다.

- **UA features (15개)**: 앱 오픈 전 광고 여정 데이터. Airbridge DB에서 조회.
- **In-app features (10개)**: 앱 오픈 후 첫 5분 행동 데이터. SDK에서 자동 수집.

D3 Purchase, D3 Churn 모델은 25개 전부 사용.  
CATE 모델도 25개 전부 사용 (trigger 반응 예측).

In [2]:
# UA features (15개) — 광고 여정
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

# In-app features (10개) — 첫 5분 행동
INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

ALL_FEATURES = UA_FEATURES + INAPP_FEATURES
TRIGGERS = ['price_appeal', 'social_proof', 'scarcity', 'novelty']

print(f'UA: {len(UA_FEATURES)}개 / In-app: {len(INAPP_FEATURES)}개 / 전체: {len(ALL_FEATURES)}개')
print(f'Trigger 종류: {TRIGGERS}')

UA: 15개 / In-app: 10개 / 전체: 25개
Trigger 종류: ['price_appeal', 'social_proof', 'scarcity', 'novelty']


---
## Step 3: Factor Analysis — UA 15개 → 6개 잠재 차원

UA 15개 feature를 6개 핵심 차원으로 압축합니다.  
"이 유저의 광고 여정이 어떤 성격인지"를 요약하는 거예요.

6개 차원: 광고 강도, DA 채널, 수동 노출, 채널 다양성, 시간 집중도, 최근성

**이건 연구용/API 응답용이고, 예측 모델에는 원본 25개 feature를 그대로 씁니다.**  
파라미터가 거의 안 바뀌어서 매주 돌리긴 하지만 실질적 변화는 거의 없음.

In [ ]:
# Organic 유저 제외 (UA가 전부 0이라 FA에 넣으면 안 됨)
paid_mask = df['trackinglink_count'] > 0
X_ua = df.loc[paid_mask, UA_FEATURES].values

# 정규화 (z-score)
scaler = StandardScaler()
X_ua_scaled = scaler.fit_transform(X_ua)

# Factor Analysis (6개 차원 추출)
fa = FactorAnalysis(n_components=6, rotation='varimax', random_state=42)
fa.fit(X_ua_scaled)

loading_matrix = fa.components_.T  # (15, 6)

factor_names = [
    'ad_intensity',       # 광고 강도
    'display_ad',         # DA 채널
    'passive_exposure',   # 수동 노출
    'channel_diversity',  # 채널 다양성
    'time_concentration', # 시간 집중도
    'recency'             # 최근성
]

# Loading matrix 출력
print('Loading Matrix (어떤 feature가 어떤 차원에 기여하는지):')
print(pd.DataFrame(loading_matrix, index=UA_FEATURES, columns=factor_names).round(2))

# 파라미터 저장
fa_params = {
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
    'loading_matrix': loading_matrix.tolist(),
    'feature_names': UA_FEATURES,
    'factor_names': factor_names
}

with open(f'../models/{APP_ID}/fa_params.json', 'w') as f:
    json.dump(fa_params, f, indent=2)

print(f'\n../models/{APP_ID}/fa_params.json 저장 완료!')
print(f'  서버에서 하는 계산: (UA 15개 - mean) / std @ loading_matrix = 6개 스코어')

---
## Step 4: D3 Purchase 모델

25개 feature로 "3일 내 구매 여부"를 예측합니다.

**전체 유저(1000명)로 학습합니다.** 구매 예측은 trigger 배정과 무관하니까  
is_random=0인 유저도 is_random=1인 유저도 전부 쓸 수 있음.

In [ ]:
# 전체 유저로 학습
X_all = df[ALL_FEATURES].values
y_purchase = df['d3_purchase'].values

purchase_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
purchase_model.fit(X_all, y_purchase)

# 성능 확인 (5-fold CV)
scores_purchase = cross_val_score(purchase_model, X_all, y_purchase, cv=5, scoring='roc_auc')
print(f'D3 Purchase AUC: {scores_purchase.mean():.3f} (±{scores_purchase.std():.3f})')
print(f'  → 0.5=동전던지기, 0.8+=실무에서 쓸만함')

# Top/Bottom 비교
pred_proba = purchase_model.predict_proba(X_all)[:, 1]
df_eval = pd.DataFrame({'pred': pred_proba, 'actual': y_purchase})
df_eval['group'] = pd.qcut(df_eval['pred'], 5, labels=['하위20%', '하위40%', '중간', '상위40%', '상위20%'], duplicates='drop')

print(f'\n=== 예측 등급별 실제 구매율 ===')
result = df_eval.groupby('group')['actual'].agg(['mean', 'count'])
result.columns = ['실제 구매율', '유저 수']
for idx, row in result.iterrows():
    print(f'  {idx}: {row["실제 구매율"]:.1%} (n={row["유저 수"]:.0f})')

top = df_eval[df_eval['group'] == '상위20%']['actual'].mean()
bot = df_eval[df_eval['group'] == '하위20%']['actual'].mean()
print(f'\n  상위 20%: {top:.1%} / 하위 20%: {bot:.1%} / 비율: {top/max(bot,0.001):.1f}x')

# 저장
purchase_data = {'model': purchase_model, 'feature_names': ALL_FEATURES}
with open(f'../models/{APP_ID}/d3_purchase_model.pkl', 'wb') as f:
    pickle.dump(purchase_data, f)
print(f'\n../models/{APP_ID}/d3_purchase_model.pkl 저장 완료!')

---
## Step 5: D3 Churn 모델

동일한 25개 feature로 "3일 내 이탈 여부"를 예측합니다.  
구매 모델과 구조가 완전히 같고 label만 다릅니다. 역시 **전체 유저**로 학습.

In [ ]:
y_churn = df['d3_churn'].values

churn_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
churn_model.fit(X_all, y_churn)

# 성능 확인
scores_churn = cross_val_score(churn_model, X_all, y_churn, cv=5, scoring='roc_auc')
print(f'D3 Churn AUC: {scores_churn.mean():.3f} (±{scores_churn.std():.3f})')

# Top/Bottom 비교
pred_churn = churn_model.predict_proba(X_all)[:, 1]
df_eval_c = pd.DataFrame({'pred': pred_churn, 'actual': y_churn})
df_eval_c['group'] = pd.qcut(df_eval_c['pred'], 5, labels=['하위20%', '하위40%', '중간', '상위40%', '상위20%'], duplicates='drop')

print(f'\n=== 예측 등급별 실제 이탈율 ===')
result_c = df_eval_c.groupby('group')['actual'].agg(['mean', 'count'])
result_c.columns = ['실제 이탈율', '유저 수']
for idx, row in result_c.iterrows():
    print(f'  {idx}: {row["실제 이탈율"]:.1%} (n={row["유저 수"]:.0f})')

# 저장
churn_data = {'model': churn_model, 'feature_names': ALL_FEATURES}
with open(f'../models/{APP_ID}/d3_churn_model.pkl', 'wb') as f:
    pickle.dump(churn_data, f)
print(f'\n../models/{APP_ID}/d3_churn_model.pkl 저장 완료!')

---
## Step 6: CATE 모델 (Trigger 반응 예측) — S-Learner

**핵심 포인트: `is_random=1`인 유저만 사용합니다.**

왜? 80% 모델 추천 유저는 이미 "잘 맞을 것 같은 trigger"를 받았기 때문에 편향되어 있음.  
이 유저들로 학습하면 Simpson's Paradox로 인과 추정이 왜곡됩니다.

20% 랜덤 유저만이 진짜 RCT 데이터이고, CATE 학습에 쓸 수 있음.

**S-Learner 방식**: 하나의 모델에 trigger를 one-hot feature로 포함하여 학습.  
새 유저가 오면 4개 trigger one-hot을 바꿔가며 예측, 가장 클릭 확률이 높은 trigger를 추천.

T-Learner(trigger별 별도 모델) 대비 장점:
- 모든 데이터를 하나의 모델이 공유 → 소규모 데이터에서 안정적
- trigger 간 feature 효과를 공유할 수 있음

In [ ]:
# ⚠️ 반드시 랜덤 배정 유저만 사용! (is_random=1)
# 80% 모델 추천 유저를 포함하면 Simpson's Paradox로 인과 추정이 왜곡됩니다.
df_cate = df[df['is_random'] == 1].copy()
assert len(df_cate) > 0, "랜덤 배정 유저가 없습니다!"
print(f'⚠️ CATE 학습: 랜덤 유저 {len(df_cate)}명만 사용 (모델 추천 유저 {len(df) - len(df_cate)}명 제외)')
print()

# S-Learner: 하나의 모델, trigger를 one-hot feature로 포함
# 각 유저의 row: [25개 feature, trigger_onehot(4)] → modal_clicked
X_rows = []
y_rows = []
for _, row in df_cate.iterrows():
    features = row[ALL_FEATURES].values.astype(float)
    trigger = row['assigned_trigger']
    trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
    X_rows.append(np.concatenate([features, trigger_onehot]))
    y_rows.append(row['modal_clicked'])

X_cate = np.array(X_rows)
y_cate = np.array(y_rows)

print(f'S-Learner 학습 데이터: {X_cate.shape[0]}행 x {X_cate.shape[1]}열 (25 features + 4 trigger one-hot)')
print()

# 단일 모델 학습
cate_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=10, random_state=42
)
cate_model.fit(X_cate, y_cate)

# ATE 분석 (랜덤 유저 기준)
print(f'=== ATE 분석 (랜덤 유저 기준) ===')
overall_rate = df_cate['modal_clicked'].mean()
print(f'전체 클릭률: {overall_rate:.1%}')
print()
print(f'{"Trigger":15s} | {"클릭률":>8s} | {"vs 평균":>8s} | {"p-value":>10s}')
print('-' * 55)

for trigger in TRIGGERS:
    mask = df_cate['assigned_trigger'] == trigger
    treat_rate = df_cate.loc[mask, 'modal_clicked'].mean()
    others = df_cate.loc[~mask, 'modal_clicked']
    treat = df_cate.loc[mask, 'modal_clicked']
    _, p_val = stats.ttest_ind(treat, others)
    diff = treat_rate - overall_rate
    print(f'{trigger:15s} | {treat_rate:>7.1%} | {diff:>+7.1%} | {p_val:>10.4f}')

In [ ]:
# CATE 모델 저장 (S-Learner: 단일 모델)
cate_data = {
    'model': cate_model,
    'treatment_triggers': TRIGGERS,
    'feature_names': ALL_FEATURES,
}

with open(f'../models/{APP_ID}/cate_model.pkl', 'wb') as f:
    pickle.dump(cate_data, f)

size = os.path.getsize(f'../models/{APP_ID}/cate_model.pkl') / 1024 / 1024
print(f'../models/{APP_ID}/cate_model.pkl 저장 완료! ({size:.1f} MB)')
print(f'  S-Learner: 단일 모델 (trigger를 one-hot feature로 포함)')
print(f'  학습 데이터: 랜덤 유저 {len(df_cate)}명만 사용')

---
## Step 7: 최종 확인

서버에 업로드할 파일 4개를 확인합니다.

In [ ]:
print(f'=== 서버에 업로드할 파일 ({APP_ID}) ===')
print()

files = [
    ('fa_params.json', 'FA 파라미터 (유저 성향 차원, 거의 안 바뀜)'),
    ('d3_purchase_model.pkl', f'D3 구매 예측 (AUC={scores_purchase.mean():.3f}, 전체 {len(df)}명으로 학습)'),
    ('d3_churn_model.pkl', f'D3 이탈 예측 (AUC={scores_churn.mean():.3f}, 전체 {len(df)}명으로 학습)'),
    ('cate_model.pkl', f'CATE 모델 (랜덤 유저 {len(df_random)}명으로 학습)'),
]

for fname, desc in files:
    path = f'../models/{APP_ID}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        unit = 'KB'
        if size > 1024:
            size /= 1024
            unit = 'MB'
        print(f'  [OK] {fname:30s} ({size:.1f} {unit}) — {desc}')
    else:
        print(f'  [!!] {fname} — 생성 안 됨!')

---
## Step 8: 테스트 — 유저 1명 예측

서버에서 API 호출이 들어왔을 때 일어나는 일을 그대로 재현합니다.  
3개 모델 전부 사용해서 완전한 API 응답을 만들어봅니다.

In [ ]:
# 유저 1명 선택 (paid 유저)
user = df[df['trackinglink_count'] > 0].iloc[0]
print(f'===== 유저 {user["user_id"]} — 전체 예측 결과 =====')
print()

# 1. Latent dimensions (FA)
ua_raw = user[UA_FEATURES].values.astype(float)
ua_norm = (ua_raw - np.array(fa_params['mean'])) / np.array(fa_params['std'])
latent = ua_norm @ np.array(fa_params['loading_matrix'])

print('1) Latent Dimensions (광고 여정 성향):')
latent_dict = {}
for name, val in zip(factor_names, latent):
    latent_dict[name] = round(float(val), 4)
    print(f'   {name}: {val:.2f}')

# 2. D3 Purchase / Churn
features = user[ALL_FEATURES].values.astype(float).reshape(1, -1)
purchase_prob = purchase_model.predict_proba(features)[0][1]
churn_prob = churn_model.predict_proba(features)[0][1]

print(f'\n2) D3 예측:')
print(f'   구매 확률: {purchase_prob:.2f}')
print(f'   이탈 확률: {churn_prob:.2f}')

# 3. CATE — Trigger scores (S-Learner)
print(f'\n3) Trigger Scores (S-Learner CATE):')
user_features = user[ALL_FEATURES].values.astype(float)
trigger_scores = {}
for trigger in TRIGGERS:
    trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
    x = np.concatenate([user_features, trigger_onehot]).reshape(1, -1)
    prob = cate_model.predict_proba(x)[0][1]
    trigger_scores[trigger] = round(float(prob), 4)
    print(f'   {trigger:15s}: {prob:.4f}')

best_trigger = max(trigger_scores, key=trigger_scores.get)
print(f'   → best_trigger: {best_trigger}')

# 4. 최종 API 응답
print(f'\n===== API 응답 =====')
response = {
    'user_id': user['user_id'],
    'best_trigger': best_trigger,
    'trigger_scores': trigger_scores,
    'd3_purchase_prob': round(float(purchase_prob), 4),
    'd3_churn_prob': round(float(churn_prob), 4),
}
print(json.dumps(response, indent=2, ensure_ascii=False))

---
## Step 9: 서버에 모델 업로드

학습한 모델 파일을 서버에 API로 업로드합니다.  
서버가 자동으로 모델을 리로드하므로 재배포 없이 바로 반영됩니다.

**SERVER_URL을 실제 서버 주소로 바꿔주세요.**

In [ ]:
import requests

# 서버 주소 (로컬 테스트: http://localhost:8000, Railway: https://your-app.railway.app)
SERVER_URL = "http://localhost:8000"

# 업로드할 파일 목록
model_files = [
    f"../models/{APP_ID}/fa_params.json",
    f"../models/{APP_ID}/d3_purchase_model.pkl",
    f"../models/{APP_ID}/d3_churn_model.pkl",
    f"../models/{APP_ID}/cate_model.pkl",
]

print(f"=== 서버에 모델 업로드 ({APP_ID}) ===")
print(f"서버: {SERVER_URL}")
print()

for filepath in model_files:
    filename = os.path.basename(filepath)
    
    if not os.path.exists(filepath):
        print(f"  [SKIP] {filename} — 파일 없음")
        continue
    
    size_kb = os.path.getsize(filepath) / 1024
    
    try:
        with open(filepath, "rb") as f:
            resp = requests.post(
                f"{SERVER_URL}/v1/models/{APP_ID}/upload",
                files={"file": (filename, f)},
            )
        
        if resp.status_code == 200:
            result = resp.json()
            print(f"  [OK] {filename:30s} ({size_kb:.1f} KB) → {result['mode']}")
        else:
            print(f"  [ERROR] {filename}: {resp.status_code} {resp.text}")
    except requests.ConnectionError:
        print(f"  [ERROR] 서버에 연결할 수 없습니다. SERVER_URL을 확인하세요: {SERVER_URL}")
        break

# 업로드 후 서버 상태 확인
print()
try:
    health = requests.get(f"{SERVER_URL}/health").json()
    print(f"서버 상태: {health['loaded_apps']}")
except:
    print("서버 상태 확인 실패")